In [2]:
import numpy as np
import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

from sklearn.metrics import accuracy_score, classification_report

import joblib
import os

In [3]:
# Load the Breast Cancer dataset

data = load_breast_cancer()

X = pd.DataFrame(
    data.data,
    columns=data.feature_names
)

y = pd.Series(
    data.target,
    name="target"
)

print("Dataset shape:", X.shape)
print("\nFeatures:")
print(X.head())

print("\nTarget classes:")
print(data.target_names)

print("\nClass distribution:")
print(y.value_counts())

Dataset shape: (569, 30)

Features:
   mean radius  mean texture  mean perimeter  mean area  mean smoothness  \
0        17.99         10.38          122.80     1001.0          0.11840   
1        20.57         17.77          132.90     1326.0          0.08474   
2        19.69         21.25          130.00     1203.0          0.10960   
3        11.42         20.38           77.58      386.1          0.14250   
4        20.29         14.34          135.10     1297.0          0.10030   

   mean compactness  mean concavity  mean concave points  mean symmetry  \
0           0.27760          0.3001              0.14710         0.2419   
1           0.07864          0.0869              0.07017         0.1812   
2           0.15990          0.1974              0.12790         0.2069   
3           0.28390          0.2414              0.10520         0.2597   
4           0.13280          0.1980              0.10430         0.1809   

   mean fractal dimension  ...  worst radius  worst text

In [4]:
# Split the dataset into training and testing sets

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)
print("Training labels:", y_train.shape)
print("Testing labels:", y_test.shape)

Training data: (455, 30)
Testing data: (114, 30)
Training labels: (455,)
Testing labels: (114,)


In [5]:
# Create a machine learning pipeline
# StandardScaler scales the features
# LogisticRegression performs classification

model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=1000))
])

# Train the model
model.fit(X_train, y_train)

print("Model trained successfully!")

Model trained successfully!


In [6]:
# Make predictions on the test data

y_pred = model.predict(X_test)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", round(accuracy, 4))

# Display detailed classification metrics
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=data.target_names))

Accuracy: 0.9825

Classification Report:
              precision    recall  f1-score   support

   malignant       0.98      0.98      0.98        42
      benign       0.99      0.99      0.99        72

    accuracy                           0.98       114
   macro avg       0.98      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114



In [7]:
# Save the trained model using joblib

model_path = "breast_cancer_model.joblib"

joblib.dump(model, model_path)

print("Model saved successfully!")
print("File:", model_path)
print("File exists:", os.path.exists(model_path))


Model saved successfully!
File: breast_cancer_model.joblib
File exists: True


In [8]:
# Load the saved model

loaded_model = joblib.load("breast_cancer_model.joblib")

print("Model loaded successfully!")

# Take one sample from the test dataset
sample = X_test.iloc[[0]]

# Make prediction
prediction = loaded_model.predict(sample)[0]

# Get prediction probability
probability = loaded_model.predict_proba(sample)[0].max()

print("\nPrediction:", prediction)
print("Predicted Class:", data.target_names[prediction])
print("Confidence:", round(probability, 4))

Model loaded successfully!

Prediction: 0
Predicted Class: malignant
Confidence: 1.0


In [9]:
# Create FastAPI application

fastapi_code = """
from fastapi import FastAPI
from pydantic import BaseModel
import pandas as pd
import joblib

app = FastAPI(title="Breast Cancer Prediction API")

# Load trained model
model = joblib.load("breast_cancer_model.joblib")


class PredictionRequest(BaseModel):
    features: list[float]


@app.get("/")
def home():
    return {
        "message": "Breast Cancer Prediction API is running"
    }


@app.post("/predict")
def predict(request: PredictionRequest):

    input_data = pd.DataFrame([request.features])

    prediction = int(model.predict(input_data)[0])
    confidence = float(model.predict_proba(input_data).max())

    return {
        "prediction": prediction,
        "confidence": round(confidence, 4)
    }
"""

# Save FastAPI code
with open("main.py", "w") as file:
    file.write(fastapi_code)

print("main.py created successfully!")
print("\nFastAPI endpoint:")
print("GET  /")
print("POST /predict")

main.py created successfully!

FastAPI endpoint:
GET  /
POST /predict
